# 07 · Speech-stage metrics, ERD & post-stage effects

Backs the report's §3 (speech artifacts), §4.2 (reading alpha ERD), §4.3 (blinks/GSR) and
§4.4 (no post-speech hypoxia-like slowing).  Robust 2-s windows; artifact rate reported.

In [1]:
import warnings, pathlib, sys
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
_c = pathlib.Path.cwd()
REPO = next(p for p in [_c] + list(_c.parents) if (p / "analysis").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from analysis import dataset, eegkit as K
DATA_DIR = None
recs = dataset.discover(DATA_DIR)
_gi, LABEL = {}, {}
for _r in recs:
    _g = _r["group"]; _gi[_g] = _gi.get(_g, 0) + 1
    LABEL[_r["key"]] = ("exp" if _g == "E" else "ctrl") + str(_gi[_g])
def lbl(r): return LABEL[r["key"]]
FIG = REPO / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
def ec_window(r):
    st = K.get_stages(K.load(r["path"]), r["key"])
    return [s for s in st if s[0].startswith("rest_close")][0]
def get_iaf(r):
    raw = K.load(r["path"]); ec = ec_window(r)
    _, live = K.get(raw, ec[1], ec[2]); return K.iaf(raw, live, ec[1], ec[2])
def stage_of(r, nm):
    st = K.get_stages(K.load(r["path"]), r["key"]); return [s for s in st if s[0] == nm]
print(f"discovered {len(recs)} recordings ->", {lbl(r): r["group"] for r in recs})


discovered 4 recordings -> {'exp1': 'E', 'exp2': 'E', 'ctrl1': 'C', 'ctrl2': 'C'}


In [2]:
_BLINK_THR={}
def blink_thr(raw):
    key=lbl_id=id(raw)
    if key not in _BLINK_THR:
        fs=raw.info["sfreq"]
        fp=raw.ch_names.index("Fp1") if "Fp1" in raw.ch_names else raw.ch_names.index("Fp2")
        d=np.abs(K.bp(raw.get_data()[fp]*1e6, fs, 1, 10))
        _BLINK_THR[key]=max(25.0, 0.8*np.percentile(d,99.5))
    return _BLINK_THR[key]
def stage_metrics(raw, st, IAF):
    fs=raw.info["sfreq"]
    x_uV = raw.get_data()*1e6
    xi, live = K.get(raw, *st)
    i0,i1=int(st[0]*fs),int(st[1]*fs)
    W=int(2*fs); n=xi.shape[1]//W
    X=xi[:, :n*W].reshape(xi.shape[0], n, W)
    mx=np.abs(X).max(axis=(0,2))
    thr=max(300.0, np.percentile(mx,60)); art=mx>thr; clean=~art
    if clean.sum()<3: clean = mx < np.median(mx)+1e-12
    cen=[live.index(c) for c in ("C3","Cz","C4") if c in live]
    fz=live.index("Fz") if "Fz" in live else (live.index("FCz") if "FCz" in live else live.index("Cz"))
    Xc=X[cen][:,clean].reshape(len(cen),-1)
    alpha=K.band_pow(Xc,fs,IAF-2,IAF+2).mean(); beta=K.band_pow(Xc,fs,13,30).mean()
    th=K.band_pow(xi[[fz]],fs,4,8).mean()
    fp=live.index("Fp1") if "Fp1" in live else live.index("Fp2")
    d=K.bp(xi[fp], fs, 1, 10)
    from scipy.signal import find_peaks
    pk,_=find_peaks(d,distance=int(0.4*fs),prominence=blink_thr(raw))
    blink=len(pk)/((st[1]-st[0])/60.0)
    g=x_uV[raw.ch_names.index("GSR")][i0:i1]
    return dict(alpha=alpha,beta=beta,fz_theta=th,blink=blink,art=art.mean(),gsr=np.mean(g)/1000)
pct=lambda a,b:100*(a/b-1) if b else np.nan
for r in recs:
    raw=K.load(r["path"]); st={n:(a,b) for n,a,b in K.get_stages(raw,r["key"])}
    ec = st.get("rest_close1") or st.get("rest_close4")
    _,live=K.get(raw,*ec); IAF=K.iaf(raw,live,*ec)
    m={n:stage_metrics(raw,ab,IAF) for n,ab in st.items()}
    print("="*80); print(f"{lbl(r)}  IAF {IAF:.1f}")
    print(f"{'stage':<12}{'C alpha':>8}{'C beta':>8}{'Fz th':>8}{'blink/m':>8}{'art%':>7}{'GSR mV':>8}")
    for n,v in m.items():
        print(f"{n:<12}{v['alpha']:8.2f}{v['beta']:8.2f}{v['fz_theta']:8.2f}{v['blink']:8.1f}{v['art']*100:6.1f}%{v['gsr']:8.1f}")
    for act,ref in (("speech","rest_open3"),("reading","rest_open2")):
        if act in m and ref in m:
            print(f"  ERD {act} vs {ref}: "+ " ".join(f"{k}:{pct(m[act][k],m[ref][k]):+.0f}%" for k in ("alpha","beta","fz_theta")))
    for post,pre in (("rest_open4","rest_open3"),("rest_open3","rest_open2"),("rest_open2","rest_open1")):
        if post in m and pre in m:
            print(f"  post {post} vs {pre}: "+ " ".join(f"{k}:{pct(m[post][k],m[pre][k]):+.0f}%" for k in ("alpha","beta","fz_theta","blink","gsr")))
print("\nReading alpha ERD ~ -40% everywhere; speech fmTheta up in PWS. Post-speech: no slowing (<=1%).")


exp1  IAF 9.6
stage        C alpha  C beta   Fz th blink/m   art%  GSR mV
rest_open1      2.13    5.62    5.80    53.3  40.4%   186.8
rest_close1    18.49    8.61    5.43    40.7  40.4%   177.5
hypervent       3.49    6.38   10.39    76.0  40.4%   265.7
rest_open2      5.05    9.62    7.91    58.7  40.4%   194.0
reading         2.69    3.88   14.14    72.0  39.6%   277.8
rest_open3      1.94    3.52    6.16    82.0  40.4%   248.2
speech          3.17    3.95   13.06    82.1  40.0%   293.7
rest_open4      1.98    3.82    6.03    78.3  40.0%   251.5
rest_close4    14.83    7.03    5.03    35.8  40.0%   239.0
  ERD speech vs rest_open3: alpha:+63% beta:+12% fz_theta:+112%
  ERD reading vs rest_open2: alpha:-47% beta:-60% fz_theta:+79%
  post rest_open4 vs rest_open3: alpha:+2% beta:+8% fz_theta:-2% blink:-4% gsr:+1%
  post rest_open3 vs rest_open2: alpha:-61% beta:-63% fz_theta:-22% blink:+40% gsr:+28%
  post rest_open2 vs rest_open1: alpha:+137% beta:+71% fz_theta:+36% blink:+10% gsr:+4%

exp2  IAF 9.8
stage        C alpha  C beta   Fz th blink/m   art%  GSR mV
rest_open1      3.17    6.30    7.16    70.3  40.4%    -0.3
rest_close1    32.74   10.45    6.19    30.7  40.4%    -0.1
hypervent       2.66    4.58    7.83    68.0  40.4%    -0.1
rest_open2      5.00    7.82    7.33    50.0  40.0%    -0.1
reading         3.25    4.38   11.21    65.7  40.0%    -0.2
speech          3.17    3.82   11.41    78.2  39.9%    -0.1
rest_open4      2.57    5.19    6.71    72.7  40.4%    -0.2
rest_close4    18.30    8.37    6.09    31.2  40.0%    -0.1
  ERD reading vs rest_open2: alpha:-35% beta:-44% fz_theta:+53%
  post rest_open2 vs rest_open1: alpha:+58% beta:+24% fz_theta:+2% blink:-29% gsr:-79%


ctrl1  IAF 9.8
stage        C alpha  C beta   Fz th blink/m   art%  GSR mV
rest_open1      3.33    1.86    3.31    17.6   0.0%    -0.2
rest_close1    11.65    2.35    4.37    21.2  40.0%    -0.1
hypervent       4.16    2.03    3.97    24.3  40.0%     0.3
rest_open2      6.35    2.51    3.92    26.0  35.6%    -0.2
reading         3.73    2.67    4.13    20.7  40.0%     0.1
rest_open3      3.90    2.20    4.02    14.3  40.0%    -0.1
speech          3.97    4.55    3.87    32.2  40.2%     0.1
  ERD speech vs rest_open3: alpha:+2% beta:+106% fz_theta:-4%
  ERD reading vs rest_open2: alpha:-41% beta:+7% fz_theta:+6%
  post rest_open3 vs rest_open2: alpha:-39% beta:-12% fz_theta:+3% blink:-45% gsr:-22%
  post rest_open2 vs rest_open1: alpha:+91% beta:+35% fz_theta:+18% blink:+48% gsr:-24%


ctrl2  IAF 9.9
stage        C alpha  C beta   Fz th blink/m   art%  GSR mV
speech          4.86    3.33    4.54    54.3  39.8%    -0.2
rest_open4      5.94    3.07    3.54    20.0  40.4%    -0.1
rest_close4    10.93    2.61    4.15    20.6  40.0%    -0.0

Reading alpha ERD ~ -40% everywhere; speech fmTheta up in PWS. Post-speech: no slowing (<=1%).
